### Ingestión de la carpeta "movie_company"
#### Este notebook es un clon del 10-Ingestion File movie_company de la carpeta ingestion con algunas cosas modificadas para que sea carga incremental, las cosas modificadas tienen el comentario al lado, básicamente lo que se modifica es:
1. Se agrega el widget v_file_date con el nombre de las carpetas de carga incremental del contenedor bronze-inc
2. Se modifica la creación del movie_companies_df, usando la variable del contenedor de bronze incremental y apuntando al widget v_file_date
3. Se agrega la columna file_date
4. Se elimina el paso "Escribir datos en el datalake en formato Parquet"
5. Se cambia la forma de crear la managed table en el ultimo paso, el esquema, el modo de escritura, la particion, y además se agrega un paso previo que elimine registros en caso de duplicados

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")






###Paso 1 - Leer los archivos CSV usando "DataFrameReader" de Spark
####La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

movie_companies_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("companyId", IntegerType(), True)
])


In [0]:
movie_companies_df = spark.read.schema(movie_companies_schema).csv(f"{bronze_inc_folder_path}/{v_file_date}/movie_company")






In [0]:
movie_companies_df.printSchema()

In [0]:
display(movie_companies_df)

In [0]:
movie_companies_df.count()

### Paso 2 - Renombrar las columnas y añadir columnas nuevas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

movie_companies_final_df = movie_companies_df \
                               .withColumnsRenamed({"movieId": "movie_id",
                                             "companyId": "company_id"})

movie_companies_final_df = add_ingestion_date(movie_companies_final_df) \
                                .withColumn("environment", lit(v_environment)) \
                                .withColumn("file_date", lit(v_file_date))


In [0]:
display(movie_companies_final_df)

### Paso 3 - Escribir datos en una managed table en el contenedor silver

In [0]:
#delete_dups("movie_silver_inc","movies_companies","file_date",f"{v_file_date}")












In [0]:
#movie_companies_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver_inc.movies_companies")

merge_condition = 'tgt.movie_id = src.movie_id and tgt.company_id = src.company_id and tgt.file_date = src.file_date'
merge_delta_lake(movie_companies_final_df,'movie_silver_inc','movies_companies',merge_condition,'file_date')


















In [0]:
%sql
SELECT file_date, COUNT(1) 
FROM movie_silver_inc.movies_companies
GROUP BY file_date

In [0]:
dbutils.notebook.exit("Success")